In [ ]:
import csv
import re
import os
import pandas as pd

def ultimate_dynamic_cleaner(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"⚠️ File not found: {input_file}")
        return

    kept_count = 0
    dropped_count = 0
    
    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8', newline='') as fout:
        
        writer = csv.writer(fout)
        
        # Read Header and determine exact expected length
        header_line = next(fin)
        reader = csv.reader([header_line])
        header = next(reader)
        header = [col.strip() for col in header]
        expected_cols = len(header)
        writer.writerow(header)
        
        # Dynamically find the indices of columns that MUST be numeric
        # This prevents errors even if column orders differ across files
        anchor_names = ['ASN', 'TTL', 'oc_8', 'dec_32', 'hex_32', 'entropy', 'hex_8', "subdomain", "len", "oc_32", "obfuscate_at_sign"]
        anchor_indices = [header.index(name) for name in anchor_names if name in header]

        for line in fin:
            # Pre-process to fix unquoted internal commas
            # A. Time delta: "4277 days, 21:07:56" -> "4277 days 21:07:56" (Remove comma)
            line = re.sub(r'(\d+\s+days),\s+(\d+:\d\d:\d\d(?:\.\d+)?)', r'\1 \2', line)
            
            # Company suffixes: "Company, Inc." -> "Company Inc." (Remove comma)
            company_suffixes = r'(Inc\.|LLC\.|Ltd\.|LLC|Inc|Ltd|UAB|S\.A\.|GmbH|Corp\.|Corporation|Co\.|Co)'
            line = re.sub(rf',\s*{company_suffixes}(?=[,\n\r])', r' \1', line, flags=re.IGNORECASE)

            # Wrap dicts and lists in double quotes to protect internal commas
            line = re.sub(r'(?<!")(defaultdict\(<class \'.*?\'>, \{.*?\}\))(?!")', r'"\1"', line)
            line = re.sub(r'(?<!")(\[.*?\])(?!")', r'"\1"', line)
            
            # Parse line into list using csv.reader
            try:
                row = next(csv.reader([line]))
            except Exception:
                dropped_count += 1
                continue
            
            # Strict Length Check (Drop if delimiters are missing)
            if len(row) != expected_cols:
                dropped_count += 1
                continue
                
            # Ensure specific columns are numbers, not shifted text
            def is_numeric(val):
                v = str(val).strip().lower()
                if v in ['', 'nan', 'none', '-1', '-1.0']: return True
                try:
                    float(v)
                    return True
                except ValueError:
                    return False

            valid = True
            for idx in anchor_indices:
                if not is_numeric(row[idx]):
                    valid = False
                    break
            
            if not valid:
                dropped_count += 1
                continue

            # 6. Save clean row
            writer.writerow(row)
            kept_count += 1

    print(f"[{input_file}] Clean: {kept_count} | Dropped: {dropped_count}")


files = [
    # ('CSV_benign.csv', 'clean_benign.csv'),
    # ('CSV_malware.csv', 'clean_malware.csv'),
    # ('CSV_phishing.csv', 'clean_phishing.csv'),
    # ('CSV_spam.csv', 'clean_spam.csv')
    # ('df.csv', 'clean_df.csv')
]

print("Starting robust data cleaning pipeline...\n")
for in_file, out_file in files:
    ultimate_dynamic_cleaner(in_file, out_file)

Starting robust data cleaning pipeline...

[df.csv] ✅ Clean: 485362 | 🗑️ Dropped: 69


In [1]:
import pandas as pd 

df_b = pd.read_csv('clean_benign.csv')
df_m = pd.read_csv('clean_malware.csv')
df_p = pd.read_csv('clean_phishing.csv')
df_s = pd.read_csv('clean_spam.csv')

C:\Users\cskok\AppData\Local\Temp\ipykernel_167880\194908153.py:3: DtypeWarning: Columns (12,13,15,17,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_b = pd.read_csv('clean_benign.csv')


In [2]:
len(df_b), len(df_m), len(df_p), len(df_s)

(471929, 4640, 4535, 4327)

In [3]:
# Add a label column to each dataframe for later concatenation
# 0 = benign, 1 = malware, 2 = phishing, 3 = spam
df_b["label"] = 0
df_m["label"] = 1
df_p["label"] = 2
df_s["label"] = 3

In [ ]:
df_final = pd.concat([df_b, df_m, df_p, df_s], ignore_index=True)

print("Final Combined Shape:", df_final.shape)
print(df_final['label'].value_counts())

df_final.to_csv('df.csv', index=False)

Final Combined Shape: (485431, 39)
label
0    471929
1      4640
2      4535
3      4327
Name: count, dtype: int64


In [6]:
df_clean = pd.read_csv('clean_df.csv')

C:\Users\cskok\AppData\Local\Temp\ipykernel_154544\3868926250.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv('clean_df.csv')


In [7]:
single_value_cols = [col for col in df_clean.columns if df_clean[col].nunique(dropna=False) == 1]

print("Columns to be removed (Zero-Variance):", single_value_cols)

df_clean.drop(columns=single_value_cols, inplace=True)

# df_clean.to_csv('clean_df.csv', index=False)

Columns to be removed (Zero-Variance): ['hex_8', 'obfuscate_at_sign', 'dec_8', 'oc_8']


In [8]:
print(df_clean.columns)
for col in df_clean.columns:
    print(f"{col}: {df_clean[col].value_counts()}")
    print()


Index(['Country', 'ASN', 'TTL', 'IP', 'Domain', 'State', 'Registrant_Name',
       'Country.1', 'Creation_Date_Time', 'hex_32', 'Domain_Name',
       'Alexa_Rank', 'subdomain', 'Organization', 'len', 'longest_word',
       'oc_32', 'shortened', '1gram', 'entropy', 'Domain_Age', 'tld', 'dec_32',
       'Emails', 'numeric_percentage', 'puny_coded', 'typos', '3gram',
       'char_distribution', '2gram', 'Registrar', 'sld', 'Name_Server_Count',
       'Page_Rank', 'label'],
      dtype='object')
Country: Country
US        205052
DE         21670
CN         19576
RU         12217
VG         11719
           ...  
38363          1
62217          1
63017          1
136038         1
59078          1
Name: count, Length: 280, dtype: int64

ASN: ASN
13335.0     46938
16509.0     21190
14618.0     13205
40034.0     12807
26496.0     10868
            ...  
49485.0         1
60824.0         1
22332.0         1
11178.0         1
199930.0        1
Name: count, Length: 10159, dtype: int64

TTL: TTL
2

In [9]:
df_clean = pd.read_csv('clean_df.csv')

cols_to_drop = [
    'Country.1', 'hex_32', 'hex_8', 'dec_8', 
    'dec_32', 'puny_coded', 'oc_8', 'oc_32', 
    'shortened', 'Page_Rank', "Alexa_Rank", "typos", "Creation_Date_Time",
    "1gram", "2gram", "3gram"
]

df_final = df_clean.drop(columns=cols_to_drop, errors='ignore')

print(df_final.columns)
print(len(df_final.columns))

df_final.to_csv('clean_df.csv', index=False)

C:\Users\cskok\AppData\Local\Temp\ipykernel_154544\1811527654.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv('clean_df.csv')


Index(['Country', 'ASN', 'TTL', 'IP', 'Domain', 'State', 'Registrant_Name',
       'Domain_Name', 'subdomain', 'Organization', 'len', 'longest_word',
       'obfuscate_at_sign', 'entropy', 'Domain_Age', 'tld', 'Emails',
       'numeric_percentage', 'char_distribution', 'Registrar', 'sld',
       'Name_Server_Count', 'label'],
      dtype='object')
23


In [10]:
df = pd.read_csv('clean_df.csv')
df.columns

Index(['Country', 'ASN', 'TTL', 'IP', 'Domain', 'State', 'Registrant_Name',
       'Domain_Name', 'subdomain', 'Organization', 'len', 'longest_word',
       'obfuscate_at_sign', 'entropy', 'Domain_Age', 'tld', 'Emails',
       'numeric_percentage', 'char_distribution', 'Registrar', 'sld',
       'Name_Server_Count', 'label'],
      dtype='object')